# Prédiction de l'annulation des réservations

Notebook reproductible et exécutable de bout en bout.

**Objectif :** prédire `reservation_annulee` et produire un `modeles/submission.csv` contenant exactement :

1. `reservation_id`
2. `probabilite_annulation`
3. `reservation_annulee`

Le modèle final retenu est un **Random Forest**. Le seuil de décision est choisi uniquement sur l'ensemble de validation temporelle, puis le test est utilisé uniquement pour produire les prédictions finales.


## 1. Imports et graine aléatoire

In [ ]:
from pathlib import Path

print("Répertoire courant :", Path.cwd())

assert Path("ressources/reservations_train.csv").exists(), (
    "reservations_train.csv introuvable. "
    "Lancez le notebook depuis la racine du projet."
)

assert Path("ressources/reservations_test.csv").exists(), (
    "reservations_test.csv introuvable. "
    "Lancez le notebook depuis la racine du projet."
)

Path("modeles").mkdir(exist_ok=True)

print("✓ Structure du projet détectée.")


In [ ]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

SEED = 42
np.random.seed(SEED)

pd.set_option("display.max_columns", None)


## 2. Chargement des données

In [ ]:
TRAIN_PATH = "ressources/reservations_train.csv"
TEST_PATH = "ressources/reservations_test.csv"

df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Train :", df.shape)
print("Test  :", test_df.shape)

display(df.head())


## 3. EDA — structure et cible

In [ ]:
print("=" * 60)
print("STRUCTURE")
print("=" * 60)
print("Dimensions :", df.shape)
print("Doublons lignes :", df.duplicated().sum())
print("Doublons reservation_id :", df["reservation_id"].duplicated().sum())

print("\n" + "=" * 60)
print("CIBLE")
print("=" * 60)
display(df["reservation_annulee"].value_counts())
display(df["reservation_annulee"].value_counts(normalize=True))


In [ ]:
print("=" * 60)
print("VALEURS MANQUANTES")
print("=" * 60)

missing = (
    df.isna()
      .sum()
      .sort_values(ascending=False)
)

display(missing[missing > 0].to_frame("nombre"))


### Constat EDA

La cible est déséquilibrée : les annulations sont moins nombreuses que les réservations non annulées.

Cela justifie notamment l'utilisation du **F1-score de la classe annulation**, plutôt que de se limiter à l'accuracy.


## 4. Préparation des données

In [ ]:
df["date_reservation"] = pd.to_datetime(df["date_reservation"])
df["date_arrivee"] = pd.to_datetime(df["date_arrivee"])

test_df["date_reservation"] = pd.to_datetime(test_df["date_reservation"])
test_df["date_arrivee"] = pd.to_datetime(test_df["date_arrivee"])

# Une valeur manquante dans agent_id correspond à une réservation directe.
df["agent_id"] = df["agent_id"].fillna("direct")
test_df["agent_id"] = test_df["agent_id"].fillna("direct")

# Variables temporelles simples.
for data in (df, test_df):
    data["mois_arrivee"] = data["date_arrivee"].dt.month
    data["mois_reservation"] = data["date_reservation"].dt.month

target = "reservation_annulee"

# Colonnes non explicatives.
feature_drop = ["reservation_id", "date_reservation", "date_arrivee"]

X = df.drop(columns=[target] + feature_drop)
y = df[target]

X_test = test_df.drop(columns=feature_drop)

numeriques = [
    "categorie_hotel",
    "delai_reservation_jours",
    "nuits",
    "adultes",
    "enfants",
    "chambres",
    "prix_moyen_nuit_eur",
    "remise_pct",
    "montant_total_eur",
    "reservations_passees",
    "annulations_passees",
    "demandes_speciales",
    "modifications_reservation",
    "jours_liste_attente",
    "mois_arrivee",
    "mois_reservation"
]

categoriels = [
    "region_hotel",
    "ville",
    "type_destination",
    "hotel_id",
    "segment_client",
    "marche_origine",
    "canal_reservation",
    "moyen_transport",
    "formule_repas",
    "tarif_remboursable",
    "type_acompte",
    "client_type",
    "agent_id"
]

print("Variables numériques :", len(numeriques))
print("Variables catégorielles :", len(categoriels))


## 5. Split temporel

Le découpage respecte l'ordre temporel des réservations. Aucune donnée future n'est utilisée pour entraîner le modèle.

In [ ]:
ordre = df["date_reservation"].sort_values().index

X = X.loc[ordre]
y = y.loc[ordre]

dates_ordonnees = df.loc[ordre, "date_reservation"]

n_train = int(len(X) * 0.80)

X_train = X.iloc[:n_train]
X_val = X.iloc[n_train:]

y_train = y.iloc[:n_train]
y_val = y.iloc[n_train:]

print("=" * 60)
print("SPLIT TEMPOREL")
print("=" * 60)
print("Nombre de données :", len(X))
print("Train :", len(X_train))
print("Validation :", len(X_val))
print("Période train :", dates_ordonnees.iloc[0], "→", dates_ordonnees.iloc[n_train - 1])
print("Période validation :", dates_ordonnees.iloc[n_train], "→", dates_ordonnees.iloc[-1])


## 6. Préprocessing

In [ ]:
preprocesseur = ColumnTransformer(
    transformers=[
        (
            "numeriques",
            Pipeline([
                ("imputation", SimpleImputer(strategy="median"))
            ]),
            numeriques
        ),
        (
            "categoriels",
            Pipeline([
                ("imputation", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categoriels
        )
    ]
)


## 7. Fonction de recherche du seuil optimal

In [ ]:
def recherche_seuil(y_true, probabilites):
    seuils = np.arange(0.05, 0.951, 0.001)
    resultats = []

    for seuil in seuils:
        predictions = (probabilites >= seuil).astype(int)

        resultats.append({
            "seuil": seuil,
            "precision": precision_score(
                y_true, predictions, zero_division=0
            ),
            "recall": recall_score(
                y_true, predictions, zero_division=0
            ),
            "f1": f1_score(
                y_true, predictions, zero_division=0
            )
        })

    resultats = pd.DataFrame(resultats)
    meilleur = resultats.loc[resultats["f1"].idxmax()]

    return meilleur, resultats


## 8. Baseline — régression logistique

In [ ]:
baseline = Pipeline([
    ("preprocessing", preprocesseur),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=SEED
    ))
])

baseline.fit(X_train, y_train)

proba_baseline = baseline.predict_proba(X_val)[:, 1]

meilleur_baseline, resultats_baseline = recherche_seuil(
    y_val,
    proba_baseline
)

print("=" * 60)
print("BASELINE — REGRESSION LOGISTIQUE")
print("=" * 60)
print(f"Seuil    : {meilleur_baseline['seuil']:.3f}")
print(f"Precision : {meilleur_baseline['precision']:.4f}")
print(f"Recall    : {meilleur_baseline['recall']:.4f}")
print(f"F1        : {meilleur_baseline['f1']:.4f}")


## 9. Random Forest

In [ ]:
random_forest = Pipeline([
    ("preprocessing", preprocesseur),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        random_state=SEED,
        n_jobs=-1,
        class_weight=None
    ))
])

random_forest.fit(X_train, y_train)

proba_rf = random_forest.predict_proba(X_val)[:, 1]

meilleur_rf, resultats_rf = recherche_seuil(
    y_val,
    proba_rf
)

seuil_rf = float(meilleur_rf["seuil"])
pred_rf = (proba_rf >= seuil_rf).astype(int)

print("=" * 60)
print("RANDOM FOREST")
print("=" * 60)
print(f"Seuil    : {seuil_rf:.3f}")
print(f"Precision : {precision_score(y_val, pred_rf):.4f}")
print(f"Recall    : {recall_score(y_val, pred_rf):.4f}")
print(f"F1        : {f1_score(y_val, pred_rf):.4f}")


In [ ]:
print("Matrice de confusion :")
print(confusion_matrix(y_val, pred_rf))

print("\nClassification report :")
print(
    classification_report(
        y_val,
        pred_rf,
        zero_division=0
    )
)


## 10. Gradient Boosting

In [ ]:
preprocesseur_dense = ColumnTransformer(
    transformers=[
        (
            "numeriques",
            Pipeline([
                ("imputation", SimpleImputer(strategy="median"))
            ]),
            numeriques
        ),
        (
            "categoriels",
            Pipeline([
                ("imputation", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                ))
            ]),
            categoriels
        )
    ]
)

gradient_boosting = Pipeline([
    ("preprocessing", preprocesseur_dense),
    ("classifier", HistGradientBoostingClassifier(
        random_state=SEED
    ))
])

gradient_boosting.fit(X_train, y_train)

proba_gb = gradient_boosting.predict_proba(X_val)[:, 1]

meilleur_gb, resultats_gb = recherche_seuil(
    y_val,
    proba_gb
)

print("=" * 60)
print("GRADIENT BOOSTING")
print("=" * 60)
print(f"Seuil    : {meilleur_gb['seuil']:.3f}")
print(f"Precision : {meilleur_gb['precision']:.4f}")
print(f"Recall    : {meilleur_gb['recall']:.4f}")
print(f"F1        : {meilleur_gb['f1']:.4f}")


## 11. Comparaison des modèles

In [ ]:
comparaison = pd.DataFrame({
    "Modele": [
        "Régression logistique",
        "Random Forest",
        "Gradient Boosting"
    ],
    "F1": [
        meilleur_baseline["f1"],
        meilleur_rf["f1"],
        meilleur_gb["f1"]
    ],
    "Precision": [
        meilleur_baseline["precision"],
        meilleur_rf["precision"],
        meilleur_gb["precision"]
    ],
    "Recall": [
        meilleur_baseline["recall"],
        meilleur_rf["recall"],
        meilleur_gb["recall"]
    ],
    "Seuil": [
        meilleur_baseline["seuil"],
        meilleur_rf["seuil"],
        meilleur_gb["seuil"]
    ]
}).sort_values("F1", ascending=False)

display(comparaison)


## 12. Feature engineering

Plusieurs transformations ont été testées pendant le projet. Elles ont été conservées ou écartées selon leur impact sur le F1 de la classe annulation.

Les expérimentations n'ayant pas amélioré le F1 de la baseline Random Forest ne sont pas intégrées au modèle final.

**Conclusion :** le feature engineering testé n'a pas apporté de gain suffisant ; le modèle final reste le Random Forest de référence.


## 13. Importance des variables

In [ ]:
rf_classifier = random_forest.named_steps["classifier"]
rf_preprocessor = random_forest.named_steps["preprocessing"]

feature_names = rf_preprocessor.get_feature_names_out()
importances = rf_classifier.feature_importances_

importance_df = pd.DataFrame({
    "feature_transforme": feature_names,
    "importance": importances
})

def variable_origine(feature):
    feature = feature.split("__", 1)[-1]

    for col in categoriels:
        if feature == col or feature.startswith(col + "_"):
            return col

    return feature

importance_df["variable"] = importance_df[
    "feature_transforme"
].apply(variable_origine)

importance_variables = (
    importance_df
    .groupby("variable", as_index=False)["importance"]
    .sum()
    .sort_values("importance", ascending=False)
)

print("TOP 15 VARIABLES")
display(importance_variables.head(15))


In [ ]:
top = importance_variables.head(15).sort_values("importance")

plt.figure(figsize=(9, 6))
plt.barh(top["variable"], top["importance"])
plt.xlabel("Importance")
plt.ylabel("Variable")
plt.title("Top 15 variables — Random Forest")
plt.tight_layout()
plt.show()


## 14. Analyse des faux positifs et faux négatifs

In [ ]:
cm = confusion_matrix(y_val, pred_rf)

tn, fp, fn, tp = cm.ravel()

print("Vrais négatifs :", tn)
print("Faux positifs  :", fp)
print("Faux négatifs  :", fn)
print("Vrais positifs :", tp)


### Interprétation

- **Faux positif :** le modèle annonce une annulation alors que la réservation n'est pas annulée.
- **Faux négatif :** le modèle annonce une réservation conservée alors qu'elle sera annulée.

Dans les résultats obtenus, le nombre de faux positifs est supérieur au nombre de faux négatifs. Le seuil choisi favorise volontairement la détection des annulations afin d'obtenir un meilleur compromis précision/rappel mesuré par le F1.


## 15. Utilisation opérationnelle des probabilités

La probabilité produite par le modèle ne doit pas être interprétée comme une certitude.

Une utilisation raisonnable consiste à classer les réservations selon leur risque :

- **probabilité faible** : surveillance normale ;
- **probabilité intermédiaire** : surveillance renforcée ;
- **probabilité élevée** : possibilité d'une action préventive.

Le seuil optimal est utilisé pour produire la décision binaire demandée par le projet, tandis que la probabilité conserve une information plus fine pour une utilisation métier.


## 16. Entraînement du modèle final

Une fois le modèle et le seuil choisis, le pipeline est réentraîné sur l'ensemble des données d'entraînement disponibles. Le fichier de test reste indépendant.

In [ ]:
# Recréer le pipeline final pour l'entraîner sur toutes les données train.
modele_final = Pipeline([
    ("preprocessing", preprocesseur),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        random_state=SEED,
        n_jobs=-1,
        class_weight=None
    ))
])

modele_final.fit(X, y)

print("Modèle final entraîné sur les", len(X), "observations train.")
print(f"Seuil retenu : {seuil_rf:.3f}")


## 17. Sauvegarde du modèle final

In [ ]:
joblib.dump(
    modele_final,
    "modeles/modele_random_forest.pkl"
)

joblib.dump(
    seuil_rf,
    "modeles/seuil_random_forest.pkl"
)

print("modele_random_forest.pkl créé.")
print("seuil_random_forest.pkl créé.")


## 18. Prédiction sur le fichier test

In [ ]:
# IMPORTANT :
# X_test a été préparé plus haut à partir de reservations_test.csv.
# Le modèle final contient déjà le preprocessing.

probabilites = modele_final.predict_proba(X_test)[:, 1]

predictions = (
    probabilites >= seuil_rf
).astype(int)

submission = pd.DataFrame({
    "reservation_id": test_df["reservation_id"],
    "probabilite_annulation": probabilites,
    "reservation_annulee": predictions
})

submission.to_csv(
    "modeles/submission.csv",
    index=False
)

print("=" * 60)
print("SUBMISSION")
print("=" * 60)
print("Fichier créé : submission.csv")
print("Nombre de lignes :", len(submission))
print("\nColonnes :", list(submission.columns))
display(submission.head(10))


## 19. Vérification finale du livrable

Le fichier doit contenir exactement trois colonnes, dans l'ordre demandé par l'énoncé.

- `reservation_id`
- `probabilite_annulation`
- `reservation_annulee`


In [ ]:
assert list(submission.columns) == [
    "reservation_id",
    "probabilite_annulation",
    "reservation_annulee"
]

assert len(submission) == len(test_df)

assert submission["reservation_id"].equals(
    test_df["reservation_id"].reset_index(drop=True)
)

assert submission["probabilite_annulation"].between(0, 1).all()

assert set(submission["reservation_annulee"].unique()).issubset({0, 1})

print("✓ Trois colonnes exactement.")
print("✓ Ordre des colonnes correct.")
print("✓ Même ordre que le fichier test.")
print("✓ Probabilités comprises entre 0 et 1.")
print("✓ Décisions binaires valides.")
print("✓ submission.csv prêt pour le dépôt.")
